# song-search -> Geode port build (Windows target, x86_64)

Cross-compiles `matcool.song-search` for **Geometry Dash 2.2081 (x86_64 / Win64)** on a
Linux kernel, following the Geode docs ("Required C++ Tools" -> Linux cross-compile)
and matcool's own `linux-win-cross` gist.

Pipeline:
1. Install clang / cmake / git / ninja / lld
2. `xwin` splat the MSVC CRT + Windows SDK
3. `matcool/clang-msvc-sdk` CMake toolchain (clang-cl, target `x86_64-pc-windows-msvc`)
4. Geode CLI + Geode SDK + 2.2081 bindings
5. Write the mod source, run the pure-C++ logic test natively
6. Configure + build the `.geode`
7. Verify the resulting DLL architecture against GD's own `GeometryDash.exe`

CPU only. No GPU/TPU.

In [ ]:
%%bash
echo "================ 1. system deps ================"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq >/dev/null 2>&1 || true
apt-get install -y -qq --no-install-recommends \
    git cmake ninja-build clang clang-tools lld llvm libssl3 \
    unzip xz-utils zip ca-certificates curl gpg wget >/dev/null 2>&1 || true

echo "--- versions ---"
cmake --version | head -1
git --version
ninja --version
clang --version | head -2 | tail -1
clang-cl --version | head -1 || true
lld-link --version || true
llvm-mt --version || true
llvm-rc --version || true

# Geode SDK requires CMake >= 3.29
CMAKE_VER=$(cmake --version | head -1 | grep -oP '[0-9]+\.[0-9]+' | head -1)
echo "cmake: $CMAKE_VER"
if awk "BEGIN{exit !($CMAKE_VER < 3.29)}"; then
  echo "cmake too old -> installing 3.31 from Kitware repo"
  wget -qO - https://apt.kitware.com/keys/kitware-archive-latest.asc 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/kitware.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/kitware.gpg] https://apt.kitware.com/ubuntu noble main" \
    > /etc/apt/sources.list.d/kitware.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq cmake >/dev/null 2>&1 || true
  cmake --version | head -1
fi

# Geode requires Clang >= 19 (21+ recommended)
CLANG_VER=$(clang --version | grep -oP 'version \K[0-9]+' | head -1)
echo "clang major: ${CLANG_VER:-0}"
if [ "${CLANG_VER:-0}" -lt 19 ]; then
  echo "clang $CLANG_VER too old -> pulling LLVM 19 release tarball"
  curl -sSL -o /tmp/llvm.tar.xz \
    "https://github.com/llvm/llvm-project/releases/download/llvmorg-19.1.7/clang+llvm-19.1.7-x86_64-linux-gnu-ubuntu-18.04.tar.xz"
  tar -C /usr/local -xf /tmp/llvm.tar.xz --strip-components=1 || true
  export PATH=/usr/local/bin:$PATH
  clang --version | head -2 | tail -1
fi
echo "DEPS OK"

In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working
echo "================ 2. xwin splat (MSVC CRT + Windows SDK) ================"
XWIN_VER=0.10.0
curl -sSL -o xwin.tar.gz \
  "https://github.com/Jake-Shadle/xwin/releases/download/${XWIN_VER}/xwin-${XWIN_VER}-x86_64-unknown-linux-musl.tar.gz"
mkdir -p xwin && tar -xzf xwin.tar.gz -C xwin --strip-components=1
chmod +x xwin/xwin
export PATH=$PWD/xwin:$PATH
xwin --version

# xwin's --cache-dir persists finished downloads and skips them on the next
# run, so a retry loop accumulates progress through flaky connections
# (default --arch is x86_64, default --variant is desktop; debug libs are
# excluded because a Release build does not need them)
export XWIN_ACCEPT_LICENSE=1
for attempt in $(seq 1 40); do
  echo "=== xwin splat attempt $attempt / 40 ==="
  if xwin --accept-license --arch x86_64 --variant desktop \
          --timeout 300 --cache-dir /kaggle/working/xwin-cache \
          --output /kaggle/working/splat splat 2>&1 | tail -6; then
    echo "xwin splat SUCCEEDED"
    break
  fi
  echo "attempt $attempt incomplete, retrying (cache keeps finished files)"
  sleep 5
done

echo "--- required layout check ---"
for d in crt/include crt/lib/x86_64 sdk/include/ucrt sdk/include/um \
         sdk/include/shared sdk/include/winrt sdk/lib/ucrt/x86_64 sdk/lib/um/x86_64; do
  if [ -d "/kaggle/working/splat/$d" ]; then
    echo "  OK   $d"
  else
    echo "  MISS $d"
  fi
done
test -f /kaggle/working/splat/sdk/include/um/Windows.h || echo "WARNING: Windows.h not found"
du -sh /kaggle/working/splat || true

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
echo "================ 3. toolchain + Geode CLI + Geode SDK ================"

# matcool's clang-cl MSVC toolchain file (same author as the mod being ported)
git clone --depth 1 https://github.com/matcool/clang-msvc-sdk.git toolchain
ls toolchain

# Geode CLI (linux)
GEODE_CLI_VER=v3.9.0
curl -sSL -o geode.zip "https://github.com/geode-sdk/cli/releases/download/${GEODE_CLI_VER}/geode-cli-${GEODE_CLI_VER}-linux.zip"
unzip -q -o geode.zip -d geode-cli
chmod +x geode-cli/geode
export PATH=$PWD/geode-cli:$PATH
geode --version

# Geode SDK (shallow clone + set-path is the portable way here)
if [ ! -d sdk/.git ]; then
  git clone --depth 1 --branch main https://github.com/geode-sdk/sdk.git sdk
fi
geode sdk set-path "$PWD/sdk"
geode sdk version
echo "--- loader libs for Win64 ---"
ls "$PWD/sdk/loader/include/link/win64"

# Geode generates binding headers at configure time from the bindings repo.
# Pre-seeding a shallow clone avoids a second big fetch mid-configure.
if [ ! -d bindings/.git ]; then
  git clone --depth 1 https://github.com/geode-sdk/bindings.git bindings
fi
echo "--- bindings dirs ---"
ls bindings/bindings | head -20
ls bindings/bindings/2.2081

In [ ]:
import base64, os, pathlib

FILES = {
 "mod.json": "ewoJImdlb2RlIjogInY1LjExLjAiLAoJImdkIjogewoJCSJ3aW4iOiAiMi4yMDgxIiwKCQkiYW5kcm9pZCI6ICIqIiwKCQkibWFjIjogIioiLAoJCSJpb3MiOiAiKiIKCX0sCgkiaWQiOiAibWF0Y29vbC5zb25nLXNlYXJjaCIsCgkibmFtZSI6ICJTb25nIFNlYXJjaCIsCgkidmVyc2lvbiI6ICJ2MS4wLjAiLAoJImRldmVsb3BlciI6ICJtYXRjb29sIiwKCSJkZXNjcmlwdGlvbiI6ICJTZWFyY2ggZG93bmxvYWRlZCBzb25ncyB3aXRoIGZ1enp5IG1hdGNoaW5nLiBQb3J0IG9mIG1hdGNvb2wvc21hbGwtZ2QtbW9kcyBzb25nLXNlYXJjaCB0byBHZW9kZS4iLAoJImRlcGVuZGVuY2llcyI6IHsKCQkiZ2VvZGUubm9kZS1pZHMiOiAiPj12MS4yMC4wIgoJfQp9Cg==",
 "CMakeLists.txt": "Y21ha2VfbWluaW11bV9yZXF1aXJlZChWRVJTSU9OIDMuMjEpCnNldChDTUFLRV9DWFhfU1RBTkRBUkQgMjMpCnNldChDTUFLRV9DWFhfU1RBTkRBUkRfUkVRVUlSRUQgT04pCnNldChDTUFLRV9DWFhfVklTSUJJTElUWV9QUkVTRVQgaGlkZGVuKQoKcHJvamVjdChzb25nLXNlYXJjaCBWRVJTSU9OIDEuMC4wKQoKIyBBZGQgYWxsIHNvdXJjZSBmaWxlcyBpbnNpZGUgc3JjIChyZWN1cnNpdmVseSkKZmlsZShHTE9CX1JFQ1VSU0UgU09VUkNFUyBDT05GSUdVUkVfREVQRU5EUyBzcmMvKi5jcHApCgojIFNldCB1cCB0aGUgbW9kIGJpbmFyeQphZGRfbGlicmFyeSgke1BST0pFQ1RfTkFNRX0gU0hBUkVEICR7U09VUkNFU30pCgppZiAoTk9UIERFRklORUQgRU5We0dFT0RFX1NES30pCiAgICBtZXNzYWdlKEZBVEFMX0VSUk9SICJVbmFibGUgdG8gZmluZCBHZW9kZSBTREshIFBsZWFzZSBkZWZpbmUgR0VPREVfU0RLIGVudmlyb25tZW50IHZhcmlhYmxlIHRvIHBvaW50IHRvIEdlb2RlIikKZWxzZSgpCiAgICBtZXNzYWdlKFNUQVRVUyAiRm91bmQgR2VvZGU6ICRFTlZ7R0VPREVfU0RLfSIpCmVuZGlmKCkKCmFkZF9zdWJkaXJlY3RvcnkoJEVOVntHRU9ERV9TREt9ICR7Q01BS0VfQ1VSUkVOVF9CSU5BUllfRElSfS9nZW9kZSkKCiMgU2V0IHVwIGRlcGVuZGVuY2llcywgcmVzb3VyY2VzLCBhbmQgbGluayBHZW9kZS4Kc2V0dXBfZ2VvZGVfbW9kKCR7UFJPSkVDVF9OQU1FfSkK",
 ".gitignore": "YnVpbGQvCi50b29scy8KKi5nZW9kZQo=",
 "src/main.cpp": "...",
 "src/fts_fuzzy_match.h": "Ly8gTElDRU5TRQovLwovLyAgIFRoaXMgc29mdHdhcmUgaXMgZHVhbC1saWNlbnNlZCB0byB0aGUgcHVibGljIGRvbWFpbiBhbmQgdW5kZXIgdGhlIGZvbGxvd2luZwovLyAgIGxpY2Vuc2U6IHlvdSBhcmUgZ3JhbnRlZCBhIHBlcnBldHVhbCwgaXJyZXZvY2FibGUgbGljZW5zZSB0byBjb3B5LCBtb2RpZnksCi8vICAgcHVibGlzaCwgYW5kIGRpc3RyaWJ1dGUgdGhpcyBmaWxlIGFzIHlvdSBzZWUgZml0LgovLwovLyBWRVJTSU9OCi8vICAgMC4yLjAgICgyMDE3LTAyLTE4KSAgU2NvcmVkIG1hdGNoZXMgcGVyZm9ybSBleGhhdXN0aXZlIHNlYXJjaCBmb3IgYmVzdCBzY29yZQovLyAgIDAuMS4wICAoMjAxNi0wMy0yOCkgIEluaXRpYWwgcmVsZWFzZQovLwovLyBBVVRIT1IKLy8gICBGb3JyZXN0IFNtaXRoCi8vCi8vIE5PVEVTCi8vICAgQ29tcGlsaW5nCi8vICAgICBZb3UgTVVTVCBhZGQgJyNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OJyBiZWZvcmUgaW5jbHVkaW5nIHRoaXMgaGVhZGVyIGluIE9ORSBzb3VyY2UgZmlsZSB0byBjcmVhdGUgaW1wbGVtZW50YXRpb24uCi8vCi8vICAgZnV6enlfbWF0Y2hfc2ltcGxlKC4uLikKLy8gICAgIFJldHVybnMgdHJ1ZSBpZiBlYWNoIGNoYXJhY3RlciBpbiBwYXR0ZXJuIGlzIGZvdW5kIHNlcXVlbnRpYWxseSB3aXRoaW4gc3RyCi8vCi8vICAgZnV6enlfbWF0Y2goLi4uKQovLyAgICAgUmV0dXJucyB0cnVlIGlmIHBhdHRlcm4gaXMgZm91bmQgQU5EIGNhbGN1bGF0ZXMgYSBzY29yZS4KLy8gICAgIFBlcmZvcm1zIGV4aGF1c3RpdmUgc2VhcmNoIHZpYSByZWN1cnNpb24gdG8gZmluZCBhbGwgcG9zc2libGUgbWF0Y2hlcyBhbmQgbWF0Y2ggd2l0aCBoaWdoZXN0IHNjb3JlLgovLyAgICAgU2NvcmVzIHZhbHVlcyBoYXZlIG5vIGludHJpbnNpYyBtZWFuaW5nLiBQb3NzaWJsZSBzY29yZSByYW5nZSBpcyBub3Qgbm9ybWFsaXplZCBhbmQgdmFyaWVzIHdpdGggcGF0dGVybi4KLy8gICAgIFJlY3Vyc2lvbiBpcyBsaW1pdGVkIGludGVybmFsbHkgKGRlZmF1bHQ9MTApIHRvIHByZXZlbnQgZGVnZW5lcmF0ZSBjYXNlcyAocGF0dGVybj0iYWFhYWFhIiBzdHI9ImFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYSIpCi8vICAgICBVc2VzIHVpbnQ4X3QgZm9yIG1hdGNoIGluZGljZXMuIFRoZXJlZm9yZSBwYXR0ZXJucyBhcmUgbGltaXRlZCB0byAyNTYgY2hhcmFjdGVycy4KLy8gICAgIFNjb3JlIHN5c3RlbSBzaG91bGQgYmUgdHVuZWQgZm9yIFlPVVIgdXNlIGNhc2UuIFdvcmRzLCBzZW50ZW5jZXMsIGZpbGUgbmFtZXMsIG9yIG1ldGhvZCBuYW1lcyBhbGwgcHJlZmVyIGRpZmZlcmVudCB0dW5pbmcuCgojaWZuZGVmIEZUU19GVVpaWV9NQVRDSF9ICiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0gKCiNpbmNsdWRlIDxjc3RkaW50PiAvLyB1aW50OF90CiNpbmNsdWRlIDxjdHlwZS5oPiAvLyA6OnRvbG93ZXIsIDo6dG91cHBlcgojaW5jbHVkZSA8Y3N0cmluZz4gLy8gbWVtY3B5CgojaW5jbHVkZSA8Y3N0ZGlvPgoKLy8gUHVibGljIGludGVyZmFjZQpuYW1lc3BhY2UgZnRzIHsKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cik7CiAgICBzdGF0aWMgYm9vbCBmdXp6eV9tYXRjaChjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0ciwgaW50ICYgb3V0U2NvcmUpOwogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlLCB1aW50OF90ICogbWF0Y2hlcywgaW50IG1heE1hdGNoZXMpOwp9CgojaWZkZWYgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCm5hbWVzcGFjZSBmdHMgewoKICAgIC8vIEZvcndhcmQgZGVjbGFyYXRpb25zIGZvciAicHJpdmF0ZSIgaW1wbGVtZW50YXRpb24KICAgIG5hbWVzcGFjZSBmdXp6eV9pbnRlcm5hbCB7CiAgICAgICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2hfcmVjdXJzaXZlKGNvbnN0IGNoYXIgKiBwYXR0ZXJuLCBjb25zdCBjaGFyICogc3RyLCBpbnQgJiBvdXRTY29yZSwgY29uc3QgY2hhciAqIHN0ckJlZ2luLAogICAgICAgICAgICB1aW50OF90IGNvbnN0ICogc3JjTWF0Y2hlcywgIHVpbnQ4X3QgKiBuZXdNYXRjaGVzLCAgaW50IG1heE1hdGNoZXMsIGludCBuZXh0TWF0Y2gsCiAgICAgICAgICAgIGludCAmIHJlY3Vyc2lvbkNvdW50LCBpbnQgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFB1YmxpYyBpbnRlcmZhY2UKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cikgewogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgIHsKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpCiAgICAgICAgICAgICAgICArK3BhdHRlcm47CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgcmV0dXJuICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CiAgICB9CgogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlKSB7CgogICAgICAgIHVpbnQ4X3QgbWF0Y2hlc1syNTZdOwogICAgICAgIHJldHVybiBmdXp6eV9tYXRjaChwYXR0ZXJuLCBzdHIsIG91dFNjb3JlLCBtYXRjaGVzLCBzaXplb2YobWF0Y2hlcykpOwogICAgfQoKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoKGNoYXIgY29uc3QgKiBwYXR0ZXJuLCBjaGFyIGNvbnN0ICogc3RyLCBpbnQgJiBvdXRTY29yZSwgdWludDhfdCAqIG1hdGNoZXMsIGludCBtYXhNYXRjaGVzKSB7CiAgICAgICAgaW50IHJlY3Vyc2lvbkNvdW50ID0gMDsKICAgICAgICBpbnQgcmVjdXJzaW9uTGltaXQgPSAxMDsKCiAgICAgICAgcmV0dXJuIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyLCBvdXRTY29yZSwgc3RyLCBudWxscHRyLCBtYXRjaGVzLCBtYXhNYXRjaGVzLCAwLCByZWN1cnNpb25Db3VudCwgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFByaXZhdGUgaW1wbGVtZW50YXRpb24KICAgIHN0YXRpYyBib29sIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUoY29uc3QgY2hhciAqIHBhdHRlcm4sIGNvbnN0IGNoYXIgKiBzdHIsIGludCAmIG91dFNjb3JlLAogICAgICAgIGNvbnN0IGNoYXIgKiBzdHJCZWdpbiwgdWludDhfdCBjb25zdCAqIHNyY01hdGNoZXMsIHVpbnQ4X3QgKiBtYXRjaGVzLCBpbnQgbWF4TWF0Y2hlcywKICAgICAgICBpbnQgbmV4dE1hdGNoLCBpbnQgJiByZWN1cnNpb25Db3VudCwgaW50IHJlY3Vyc2lvbkxpbWl0KQogICAgewogICAgICAgIC8vIENvdW50IHJlY3Vyc2lvbnMKICAgICAgICArK3JlY3Vyc2lvbkNvdW50OwogICAgICAgIGlmIChyZWN1cnNpb25Db3VudCA+PSByZWN1cnNpb25MaW1pdCkKICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAvLyBEZXRlY3QgZW5kIG9mIHN0cmluZ3MKICAgICAgICBpZiAoKnBhdHRlcm4gPT0gJ1wwJyB8fCAqc3RyID09ICdcMCcpCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKCiAgICAgICAgLy8gUmVjdXJzaW9uIHBhcmFtcwogICAgICAgIGJvb2wgcmVjdXJzaXZlTWF0Y2ggPSBmYWxzZTsKICAgICAgICB1aW50OF90IGJlc3RSZWN1cnNpdmVNYXRjaGVzWzI1Nl07CiAgICAgICAgaW50IGJlc3RSZWN1cnNpdmVTY29yZSA9IDA7CgogICAgICAgIC8vIExvb3AgdGhyb3VnaCBwYXR0ZXJuIGFuZCBzdHIgbG9va2luZyBmb3IgYSBtYXRjaAogICAgICAgIGJvb2wgZmlyc3RfbWF0Y2ggPSB0cnVlOwogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgewoKICAgICAgICAgICAgLy8gRm91bmQgbWF0Y2gKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpIHsKCiAgICAgICAgICAgICAgICAvLyBTdXBwbGllZCBtYXRjaGVzIGJ1ZmZlciB3YXMgdG9vIHNob3J0CiAgICAgICAgICAgICAgICBpZiAobmV4dE1hdGNoID49IG1heE1hdGNoZXMpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAgICAgICAgIC8vICJDb3B5LW9uLVdyaXRlIiBzcmNNYXRjaGVzIGludG8gbWF0Y2hlcwogICAgICAgICAgICAgICAgaWYgKGZpcnN0X21hdGNoICYmIHNyY01hdGNoZXMpIHsKICAgICAgICAgICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgc3JjTWF0Y2hlcywgbmV4dE1hdGNoKTsKICAgICAgICAgICAgICAgICAgICBmaXJzdF9tYXRjaCA9IGZhbHNlOwogICAgICAgICAgICAgICAgfQoKICAgICAgICAgICAgICAgIC8vIFJlY3Vyc2l2ZSBjYWxsIHRoYXQgInNraXBzIiB0aGlzIG1hdGNoCiAgICAgICAgICAgICAgICB1aW50OF90IHJlY3Vyc2l2ZU1hdGNoZXNbMjU2XTsKICAgICAgICAgICAgICAgIGludCByZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgICAgIGlmIChmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyICsgMSwgcmVjdXJzaXZlU2NvcmUsIHN0ckJlZ2luLCBtYXRjaGVzLCByZWN1cnNpdmVNYXRjaGVzLCBzaXplb2YocmVjdXJzaXZlTWF0Y2hlcyksIG5leHRNYXRjaCwgcmVjdXJzaW9uQ291bnQsIHJlY3Vyc2lvbkxpbWl0KSkgewoKICAgICAgICAgICAgICAgICAgICAvLyBQaWNrIGJlc3QgcmVjdXJzaXZlIHNjb3JlCiAgICAgICAgICAgICAgICAgICAgaWYgKCFyZWN1cnNpdmVNYXRjaCB8fCByZWN1cnNpdmVTY29yZSA+IGJlc3RSZWN1cnNpdmVTY29yZSkgewogICAgICAgICAgICAgICAgICAgICAgICBtZW1jcHkoYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIHJlY3Vyc2l2ZU1hdGNoZXMsIDI1Nik7CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RSZWN1cnNpdmVTY29yZSA9IHJlY3Vyc2l2ZVNjb3JlOwogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICByZWN1cnNpdmVNYXRjaCA9IHRydWU7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQWR2YW5jZQogICAgICAgICAgICAgICAgbWF0Y2hlc1tuZXh0TWF0Y2grK10gPSAodWludDhfdCkoc3RyIC0gc3RyQmVnaW4pOwogICAgICAgICAgICAgICAgKytwYXR0ZXJuOwogICAgICAgICAgICB9CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgLy8gRGV0ZXJtaW5lIGlmIGZ1bGwgcGF0dGVybiB3YXMgbWF0Y2hlZAogICAgICAgIGJvb2wgbWF0Y2hlZCA9ICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CgogICAgICAgIC8vIENhbGN1bGF0ZSBzY29yZQogICAgICAgIGlmIChtYXRjaGVkKSB7CiAgICAgICAgICAgIGNvbnN0IGludCBzZXF1ZW50aWFsX2JvbnVzID0gMTU7ICAgICAgICAgICAgLy8gYm9udXMgZm9yIGFkamFjZW50IG1hdGNoZXMKICAgICAgICAgICAgY29uc3QgaW50IHNlcGFyYXRvcl9ib251cyA9IDMwOyAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBvY2N1cnMgYWZ0ZXIgYSBzZXBhcmF0b3IKICAgICAgICAgICAgY29uc3QgaW50IGNhbWVsX2JvbnVzID0gMzA7ICAgICAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBpcyB1cHBlcmNhc2UgYW5kIHByZXYgaXMgbG93ZXIKICAgICAgICAgICAgY29uc3QgaW50IGZpcnN0X2xldHRlcl9ib251cyA9IDE1OyAgICAgICAgICAvLyBib251cyBpZiB0aGUgZmlyc3QgbGV0dGVyIGlzIG1hdGNoZWQKCiAgICAgICAgICAgIGNvbnN0IGludCBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTU7ICAgICAgLy8gcGVuYWx0eSBhcHBsaWVkIGZvciBldmVyeSBsZXR0ZXIgaW4gc3RyIGJlZm9yZSB0aGUgZmlyc3QgbWF0Y2gKICAgICAgICAgICAgY29uc3QgaW50IG1heF9sZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTE1OyAvLyBtYXhpbXVtIHBlbmFsdHkgZm9yIGxlYWRpbmcgbGV0dGVycwogICAgICAgICAgICBjb25zdCBpbnQgdW5tYXRjaGVkX2xldHRlcl9wZW5hbHR5ID0gLTE7ICAgIC8vIHBlbmFsdHkgZm9yIGV2ZXJ5IGxldHRlciB0aGF0IGRvZXNuJ3QgbWF0dGVyCgogICAgICAgICAgICAvLyBJdGVyYXRlIHN0ciB0byBlbmQKICAgICAgICAgICAgd2hpbGUgKCpzdHIgIT0gJ1wwJykKICAgICAgICAgICAgICAgICsrc3RyOwoKICAgICAgICAgICAgLy8gSW5pdGlhbGl6ZSBzY29yZQogICAgICAgICAgICBvdXRTY29yZSA9IDEwMDsKCiAgICAgICAgICAgIC8vIEFwcGx5IGxlYWRpbmcgbGV0dGVyIHBlbmFsdHkKICAgICAgICAgICAgaW50IHBlbmFsdHkgPSBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ICogbWF0Y2hlc1swXTsKICAgICAgICAgICAgaWYgKHBlbmFsdHkgPCBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eSkKICAgICAgICAgICAgICAgIHBlbmFsdHkgPSBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eTsKICAgICAgICAgICAgb3V0U2NvcmUgKz0gcGVuYWx0eTsKCiAgICAgICAgICAgIC8vIEFwcGx5IHVubWF0Y2hlZCBwZW5hbHR5CiAgICAgICAgICAgIGludCB1bm1hdGNoZWQgPSAoaW50KShzdHIgLSBzdHJCZWdpbikgLSBuZXh0TWF0Y2g7CiAgICAgICAgICAgIG91dFNjb3JlICs9IHVubWF0Y2hlZF9sZXR0ZXJfcGVuYWx0eSAqIHVubWF0Y2hlZDsKCiAgICAgICAgICAgIC8vIEFwcGx5IG9yZGVyaW5nIGJvbnVzZXMKICAgICAgICAgICAgZm9yIChpbnQgaSA9IDA7IGkgPCBuZXh0TWF0Y2g7ICsraSkgewogICAgICAgICAgICAgICAgdWludDhfdCBjdXJySWR4ID0gbWF0Y2hlc1tpXTsKCiAgICAgICAgICAgICAgICBpZiAoaSA+IDApIHsKICAgICAgICAgICAgICAgICAgICB1aW50OF90IHByZXZJZHggPSBtYXRjaGVzW2kgLSAxXTsKCiAgICAgICAgICAgICAgICAgICAgLy8gU2VxdWVudGlhbAogICAgICAgICAgICAgICAgICAgIGlmIChjdXJySWR4ID09IChwcmV2SWR4ICsgMSkpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcXVlbnRpYWxfYm9udXM7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQ2hlY2sgZm9yIGJvbnVzZXMgYmFzZWQgb24gbmVpZ2hib3IgY2hhcmFjdGVyIHZhbHVlCiAgICAgICAgICAgICAgICBpZiAoY3VycklkeCA+IDApIHsKICAgICAgICAgICAgICAgICAgICAvLyBDYW1lbCBjYXNlCiAgICAgICAgICAgICAgICAgICAgY2hhciBuZWlnaGJvciA9IHN0ckJlZ2luW2N1cnJJZHggLSAxXTsKICAgICAgICAgICAgICAgICAgICBjaGFyIGN1cnIgPSBzdHJCZWdpbltjdXJySWR4XTsKICAgICAgICAgICAgICAgICAgICBpZiAoOjppc2xvd2VyKG5laWdoYm9yKSAmJiA6OmlzdXBwZXIoY3VycikpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGNhbWVsX2JvbnVzOwoKICAgICAgICAgICAgICAgICAgICAvLyBTZXBhcmF0b3IKICAgICAgICAgICAgICAgICAgICBib29sIG5laWdoYm9yU2VwYXJhdG9yID0gbmVpZ2hib3IgPT0gJ18nIHx8IG5laWdoYm9yID09ICcgJzsKICAgICAgICAgICAgICAgICAgICBpZiAobmVpZ2hib3JTZXBhcmF0b3IpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcGFyYXRvcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGVsc2UgewogICAgICAgICAgICAgICAgICAgIC8vIEZpcnN0IGxldHRlcgogICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGZpcnN0X2xldHRlcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgfQogICAgICAgIH0KCiAgICAgICAgLy8gUmV0dXJuIGJlc3QgcmVzdWx0CiAgICAgICAgaWYgKHJlY3Vyc2l2ZU1hdGNoICYmICghbWF0Y2hlZCB8fCBiZXN0UmVjdXJzaXZlU2NvcmUgPiBvdXRTY29yZSkpIHsKICAgICAgICAgICAgLy8gUmVjdXJzaXZlIHNjb3JlIGlzIGJldHRlciB0aGFuICJ0aGlzIgogICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIG1heE1hdGNoZXMpOwogICAgICAgICAgICBvdXRTY29yZSA9IGJlc3RSZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgcmV0dXJuIHRydWU7CiAgICAgICAgfQogICAgICAgIGVsc2UgaWYgKG1hdGNoZWQpIHsKICAgICAgICAgICAgLy8gInRoaXMiIHNjb3JlIGlzIGJldHRlciB0aGFuIHJlY3Vyc2l2ZQogICAgICAgICAgICByZXR1cm4gdHJ1ZTsKICAgICAgICB9CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIC8vIG5vIG1hdGNoCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKICAgICAgICB9CiAgICB9Cn0gLy8gbmFtZXNwYWNlIGZ0cwoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9JTVBMRU1FTlRBVElPTgoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9ICg==",
 "tests/logic_test.cpp": "Ly8gU3RhbmRhbG9uZSBsb2dpYyB0ZXN0IGZvciB0aGUgc29uZy1zZWFyY2ggcG9ydC4KLy8gTWlycm9ycyB0aGUgZnV6enkgc2VhcmNoICsgc29ydCArIHNjcm9sbC13aW5kb3cgbG9naWMgaW4gc3JjL21haW4uY3BwCi8vIHdpdGhvdXQgR2VvZGUgaGVhZGVycywgc28gaXQgY2FuIHJ1biBvbiBwbGFpbiBHQ0MuCiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCiNpbmNsdWRlICJmdHNfZnV6enlfbWF0Y2guaCIKCiNpbmNsdWRlIDxhbGdvcml0aG0+CiNpbmNsdWRlIDxjYXNzZXJ0PgojaW5jbHVkZSA8Y21hdGg+CiNpbmNsdWRlIDxpb3N0cmVhbT4KI2luY2x1ZGUgPHNzdHJlYW0+CiNpbmNsdWRlIDxzdHJpbmc+CiNpbmNsdWRlIDx2ZWN0b3I+CgpzdHJ1Y3QgRmFrZVNvbmcgeyBpbnQgaWQ7IHN0ZDo6c3RyaW5nIG5hbWU7IHN0ZDo6c3RyaW5nIGFydGlzdDsgfTsKCnN0YXRpYyBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8aW50LCBGYWtlU29uZz4+IGRvX3NlYXJjaCgKICAgIGNvbnN0IHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxzdGQ6OnN0cmluZywgRmFrZVNvbmc+PiYgZG93bmxvYWRlZCwKICAgIGNvbnN0IHN0ZDo6c3RyaW5nJiBpbnB1dCkgewogICAgc3RkOjp2ZWN0b3I8c3RkOjpwYWlyPGludCwgRmFrZVNvbmc+PiBzb25nczsKICAgIHNvbmdzLnJlc2VydmUoZG93bmxvYWRlZC5zaXplKCkpOwogICAgY29uc3QgYm9vbCBlbXB0eV9pbnB1dCA9IGlucHV0LmVtcHR5KCk7CiAgICBmb3IgKGNvbnN0IGF1dG8mIFtpbmRleF9zdHIsIHNvbmddIDogZG93bmxvYWRlZCkgewogICAgICAgIGludCBzY29yZSA9IDA7CiAgICAgICAgaWYgKCFlbXB0eV9pbnB1dCAmJiAhZnRzOjpmdXp6eV9tYXRjaChpbnB1dC5jX3N0cigpLCBpbmRleF9zdHIuY19zdHIoKSwgc2NvcmUpKSBjb250aW51ZTsKICAgICAgICBzb25ncy5wdXNoX2JhY2soe3Njb3JlLCBzb25nfSk7CiAgICB9CiAgICBpZiAoIWVtcHR5X2lucHV0ICYmIHNvbmdzLnNpemUoKSA+IDEpIHsKICAgICAgICBzdGQ6OnNvcnQoc29uZ3MuYmVnaW4oKSwgc29uZ3MuZW5kKCksIFsmXShjb25zdCBhdXRvJiBhLCBjb25zdCBhdXRvJiBiKSB7CiAgICAgICAgICAgIHJldHVybiBhLmZpcnN0ID4gYi5maXJzdDsKICAgICAgICB9KTsKICAgIH0KICAgIHJldHVybiBzb25nczsKfQoKaW50IG1haW4oKSB7CiAgICBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8c3RkOjpzdHJpbmcsIEZha2VTb25nPj4gZG93bmxvYWRlZCA9IHsKICAgICAgICB7IlN0ZXJlbyBNYWRuZXNzIEZvcmV2ZXIiLCB7MSwgIlN0ZXJlbyBNYWRuZXNzIiwgIkZvcmV2ZXIifX0sCiAgICAgICAgeyJCYWNrIG9uIFRyYWNrIERKVkkiLCB7MiwgIkJhY2sgb24gVHJhY2siLCAiREpWSSJ9fSwKICAgICAgICB7IlBvbGFyZ2Vpc3QgU3RlcCIsIHszLCAiUG9sYXJnZWlzdCIsICJTdGVwIn19LAogICAgICAgIHsiRHJ5IE91dCBESlZJIiwgezQsICJEcnkgT3V0IiwgIkRKVkkifX0sCiAgICAgICAgeyJCYXNlIEFmdGVyIEJhc2UgREpWSSIsIHs1LCAiQmFzZSBBZnRlciBCYXNlIiwgIkRKVkkifX0sCiAgICB9OwoKICAgIC8vIDEuIGVtcHR5IGlucHV0IHJldHVybnMgZXZlcnl0aGluZywgdW5zb3J0ZWQKICAgIGF1dG8gYWxsID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICIiKTsKICAgIGFzc2VydChhbGwuc2l6ZSgpID09IDUpOwoKICAgIC8vIDIuIGV4YWN0LWlzaCBxdWVyeSByYW5rcyB0aGUgcmlnaHQgc29uZyBmaXJzdAogICAgYXV0byBkanZpID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICJkanZpIik7CiAgICBhc3NlcnQoZGp2aS5zaXplKCkgPT0gMyk7CiAgICBmb3IgKGF1dG8mIFtzY29yZSwgc10gOiBkanZpKSAodm9pZClzY29yZSwgYXNzZXJ0KHMuYXJ0aXN0ID09ICJESlZJIik7CgogICAgLy8gMy4gZnV6enkgcXVlcnkgbWF0Y2hlcyBhY3Jvc3MgbmFtZSthcnRpc3QgYmxvYgogICAgYXV0byBwb2xhciA9IGRvX3NlYXJjaChkb3dubG9hZGVkLCAicG9sciIpOwogICAgYXNzZXJ0KHBvbGFyLnNpemUoKSA9PSAxICYmIHBvbGFyWzBdLnNlY29uZC5pZCA9PSAzKTsKCiAgICAvLyA0LiBubyBtYXRjaCAtPiBlbXB0eQogICAgYXNzZXJ0KGRvX3NlYXJjaChkb3dubG9hZGVkLCAienp6cXFxIikuZW1wdHkoKSk7CgogICAgLy8gNS4gc2Nyb2xsIGNsYW1wOiBtaXJyb3JzIHVwZGF0ZV9zY3JvbGxfZWxlbWVudHMgY2xhbXBpbmcKICAgIGNvbnN0ZXhwciBmbG9hdCBoZWlnaHQgPSA1NS5mOyAvLyB3aWRnZXRfaGVpZ2h0ICsgc3BhY2luZwogICAgY29uc3RleHByIGludCBuX3dpZGdldHMgPSA2OwogICAgYXV0byBjbGFtcF9zY3JvbGwgPSBbJl0oc2l6ZV90IGNvdW50LCBmbG9hdCB5KSB7CiAgICAgICAgY29uc3QgYXV0byBtID0gaGVpZ2h0ICogKHN0YXRpY19jYXN0PGludD4oY291bnQpIC0gbl93aWRnZXRzICsgMyk7CiAgICAgICAgaWYgKG0gPCAwKSByZXR1cm4gMC5mOwogICAgICAgIHJldHVybiBzdGQ6OmNsYW1wKHksIDAuZiwgbSk7CiAgICB9OwogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgwLCA5OTkuZikgPT0gMC5mKTsgICAvLyBlbXB0eSBsaXN0IHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgzLCA5OTkuZikgPT0gMC5mKTsgICAvLyBzaG9ydCBsaXN0IHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgLTUwLmYpID09IDAuZik7ICAvLyBuZWdhdGl2ZSBwaW5zIHRvIDAKICAgIGFzc2VydChjbGFtcF9zY3JvbGwoMjAsIDFlOWYpID09IGhlaWdodCAqICgyMCAtIG5fd2lkZ2V0cyArIDMpKTsgLy8gY2FwcyBhdCBtYXgKICAgIGFzc2VydChjbGFtcF9zY3JvbGwoMjAsIDEwMC5mKSA9PSAxMDAuZik7IC8vIGluLXJhbmdlIHVudG91Y2hlZAoKICAgIC8vIDYuIHJlc3VsdCBsYWJlbCBwbHVyYWxpemF0aW9uCiAgICBhdXRvIGxhYmVsID0gW10oc2l6ZV90IG4pIHsKICAgICAgICBzdGQ6OnN0cmluZ3N0cmVhbSBzOyBzIDw8IG4gPDwgIiByZXN1bHQiOyBpZiAobiAhPSAxKSBzIDw8ICJzIjsgcmV0dXJuIHMuc3RyKCk7CiAgICB9OwogICAgYXNzZXJ0KGxhYmVsKDApID09ICIwIHJlc3VsdHMiICYmIGxhYmVsKDEpID09ICIxIHJlc3VsdCIgJiYgbGFiZWwoNSkgPT0gIjUgcmVzdWx0cyIpOwoKICAgIHN0ZDo6Y291dCA8PCAiQUxMIExPR0lDIFRFU1RTIFBBU1NFRCIgPDwgc3RkOjplbmRsOwogICAgcmV0dXJuIDA7Cn0K"
}

ROOT = pathlib.Path('/kaggle/working/song-search')
for rel, b64data in FILES.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64data))
    print(f'wrote {p} ({p.stat().st_size} bytes)')

print('\n=== tree ===')
for dirpath, dirnames, filenames in os.walk(ROOT):
    for f in sorted(filenames):
        fp = os.path.join(dirpath, f)
        print(' ', os.path.relpath(fp, ROOT), os.path.getsize(fp))

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/song-search
echo "================ 5. native logic test ================"
g++ -std=c++20 -O2 -Wall -I src -o /tmp/logic_test tests/logic_test.cpp
/tmp/logic_test
echo "logic test exit: $?"

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/song-search
echo "================ 6. configure (cross-compile -> Win64) ================"
export SPLAT_DIR=/kaggle/working/splat
export TOOLCHAIN=/kaggle/working/toolchain/clang-cl-msvc.cmake
export HOST_ARCH=x86_64
export PATH=/usr/local/bin:$PATH:/kaggle/working/geode-cli
export GEODE_SDK=/kaggle/working/sdk
export GEODE_BINDINGS_REPO_PATH=/kaggle/working/bindings

rm -rf build
cmake \
  -DCMAKE_TOOLCHAIN_FILE=$TOOLCHAIN \
  -DCMAKE_BUILD_TYPE=Release \
  -DGEODE_TARGET_PLATFORM=Win64 \
  -G Ninja \
  -B build 2>&1 | tail -40
echo "CONFIGURE EXIT: ${PIPESTATUS[0]}"

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/song-search
echo "================ 7. build ================"
export PATH=/usr/local/bin:$PATH:/kaggle/working/geode-cli
export GEODE_SDK=/kaggle/working/sdk
export GEODE_BINDINGS_REPO_PATH=/kaggle/working/bindings
time cmake --build build --config Release 2>&1 | tail -60
echo "BUILD EXIT: ${PIPESTATUS[0]}"

echo "================ build artifacts ================"
find build -maxdepth 3 -type f \( -name '*.geode' -o -name '*.dll' -o -name '*.lib' \) -exec ls -la {} \;

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/song-search
echo "================ 8. verify .geode ================"
GEO=$(find build -maxdepth 3 -name '*.geode' | head -1)
if [ -z "$GEO" ]; then
  echo "NO .geode PRODUCED"
  find build -maxdepth 3 -type f | head -40
  exit 1
fi
echo "found: $GEO"
ls -la "$GEO"
echo "--- contents ---"
unzip -l "$GEO"

echo
echo "================ 9. verify DLL architecture ================"
mkdir -p /tmp/geocheck && cd /tmp/geocheck
rm -rf * .[!.]* 2>/dev/null || true
unzip -o -q /kaggle/working/song-search/$GEO
DLL=$(find . -name '*.dll' | head -1)
echo "mod dll: $DLL"
python3 - "$DLL" <<'PY'
import sys, struct
def pe_info(p):
    d = open(p,'rb').read()
    assert d[:2]==b'MZ', 'not a PE'
    e = struct.unpack_from('<I', d, 0x3c)[0]
    assert d[e:e+4]==b'PE\x00\x00'
    mach  = struct.unpack_from('<H', d, e+4)[0]
    chars = struct.unpack_from('<H', d, e+22)[0]
    subsys= struct.unpack_from('<H', d, e+92)[0]
    magic = struct.unpack_from('<H', d, e+24)[0]
    return {
      'machine':    {0x14c:'i386 (32-bit)',0x8664:'x86_64 (64-bit)',0xaa64:'arm64'}.get(mach, hex(mach)),
      'is_dll':     bool(chars & 0x2000),
      'subsystem':  {2:'GUI',3:'CUI'}.get(subsys, hex(subsys)),
      'pe_format':  'PE32+' if magic==0x20b else 'PE32',
      'size':       len(d),
    }
info = pe_info(sys.argv[1])
for k,v in info.items(): print(f'  {k}: {v}')
ok = info['machine'].startswith('x86_64') and info['is_dll'] and info['pe_format']=='PE32+'
print('  ARCH MATCHES GD 2.2081 (x86_64):', 'YES' if ok else 'NO -> MISMATCH')
PY

echo
echo "================ 10. mod.json sanity ================"
python3 -c "
import json
m=json.load(open('/kaggle/working/song-search/mod.json'))
print('id      :', m['id'])
print('version :', m['version'])
print('geode   :', m['geode'])
print('gd.win  :', m['gd']['win'])
assert m['gd']['win'] == '2.2081', 'expected 2.2081'
print('mod.json OK')
"

echo
echo "ALL DONE"